# 07b. Avaliar predictions — atribuição por `id_seq` (CSV da aplicação)

Sem depara Luis nesta rodada. Input = CSV da aplicação (mesmos `id_seq` do predict).

- **id1 = censo**, **id2 = cpf**
- Cada base tem `id_seq` 1..N independente
- Escada: modelo → resgate → baixo opcional
- Cobertura por UF: determinístico (lista ouro) + probabilístico (app); soma ≈ Censo

Ajuste paths na célula de config se mudar a aplicação.


In [ ]:
import sys
from pathlib import Path

from IPython.display import display

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

from config import (
    LISTA_OURO_AVALIACAO_ARQUIVO_TOTAL,
    MAX_CENSOS_POR_CPF,
    OUTPUT_DIR,
    cpf_norm_sql,
    drop_splink_temp_tables,
    ensure_output_dir,
    export_parquet,
    get_connection,
    lista_ouro_colunas,
    load_table_from_parquet,
    print_paths,
    require_input,
)

# --- config ---
# CSV da aplicação (id_seq 1..N por base). Pred: id1=censo, id2=cpf.
CENSO_APP_CSV = Path(
    '/data/singed/ramon.goncalves/probabilistico_output/uf_21/'
    'censo_aplicacao_splink.csv'
)
CPF_APP_CSV = Path(
    '/data/singed/ramon.goncalves/probabilistico_output/uf_21/'
    'cpf_aplicacao_splink.csv'
)
PRED_TXT = Path(
    '/data/singed/ramon.goncalves/probabilistico_output/censo_cpf/'
    'cpareia_results/pred_gz_p025_dedup.txt'
)
# Só para cobertura UF (determinístico = lista ouro; universo = Luis)
CENSO_V3_LUIS = Path(
    '/data/singed/capefe/scripts_luis/Pareamento Probabilístico/'
    'V3 20260925/censo_limpo.parquet'
)

PISO_MODELO = 0.97
PISO_RESGATE = 0.50
PISO_BAIXO = 0.25
TETO = MAX_CENSOS_POR_CPF

RUN_MODELO = True
RUN_RESGATE = True
RUN_BAIXO = False

REBUILD_APP = False
REBUILD_PRED = False
REBUILD_PARES_RESGATE = True
REBUILD_PARES_BAIXO = False
REBUILD_ATRIB = True

APP_COLS = [
    'id_seq',
    'nome_completo_phon',
    'primeiro_nome_phon',
    'ultimo_nome_phon',
    'data_nascimento',
    'ano_nascimento',
    'mes_nascimento',
    'dia_nascimento',
    'idade',
    'nome_mae_phon',
    'logradouro_norm',
    'cep',
    'cod_municipio',
    'sexo',
]

APP_CENSO_OUT = OUTPUT_DIR / '07b_app_censo.parquet'
APP_CPF_OUT = OUTPUT_DIR / '07b_app_cpf.parquet'
PRED_SEQ_OUT = OUTPUT_DIR / '07b_pred_seq.parquet'
PARES_RESGATE_OUT = OUTPUT_DIR / '07b_pares_resgate.parquet'
PARES_BAIXO_OUT = OUTPUT_DIR / '07b_pares_baixo.parquet'
ATRIB_SEQ_OUT = OUTPUT_DIR / '07b_atribuicao_idseq.parquet'
COBERTURA_UF_OUT = OUTPUT_DIR / '07b_cobertura_uf.parquet'

print_paths()
ensure_output_dir()
print('CENSO_APP_CSV:', CENSO_APP_CSV)
print('CPF_APP_CSV:  ', CPF_APP_CSV)
print('PRED_TXT:', PRED_TXT)
print('RUN:', {'modelo': RUN_MODELO, 'resgate': RUN_RESGATE, 'baixo': RUN_BAIXO})
print('REBUILD:', {
    'app': REBUILD_APP,
    'pred': REBUILD_PRED,
    'pares_resgate': REBUILD_PARES_RESGATE,
    'pares_baixo': REBUILD_PARES_BAIXO,
    'atrib': REBUILD_ATRIB,
})
print(
    f'Faixas: modelo>={PISO_MODELO} | resgate>={PISO_RESGATE} | '
    f'baixo>={PISO_BAIXO} | teto={TETO}'
)

require_input(CENSO_APP_CSV, label='CENSO_APP_CSV')
require_input(CPF_APP_CSV, label='CPF_APP_CSV')
require_input(CENSO_V3_LUIS, label='CENSO_V3_LUIS')
require_input(
    LISTA_OURO_AVALIACAO_ARQUIVO_TOTAL,
    label='LISTA_OURO_AVALIACAO_TOTAL',
)
if REBUILD_PRED or not PRED_SEQ_OUT.exists():
    require_input(PRED_TXT, label='PRED_TXT')

con = get_connection()
drop_splink_temp_tables(con)

def duck_tem_tabela(nome):
    n = con.execute(
        "SELECT COUNT(*) FROM information_schema.tables "
        f"WHERE table_schema = 'main' AND table_name = '{nome}'"
    ).fetchone()[0]
    return n > 0

def garantir_atribuicao():
    if duck_tem_tabela('atribuicao'):
        print('atribuicao: já no DuckDB |',
              f"{con.execute('SELECT COUNT(*) FROM atribuicao').fetchone()[0]:,}")
        return
    if ATRIB_SEQ_OUT.exists():
        load_table_from_parquet(con, 'atribuicao', ATRIB_SEQ_OUT)
        print('atribuicao: lida do parquet id_seq |',
              f"{con.execute('SELECT COUNT(*) FROM atribuicao').fetchone()[0]:,}")
        return
    con.execute('''
    CREATE OR REPLACE TABLE atribuicao (
        id_seq_censo VARCHAR,
        id_seq_cpf VARCHAR,
        match_probability DOUBLE,
        faixa VARCHAR,
        regra VARCHAR
    )
    ''')
    print('atribuicao: tabela vazia (nova, id_seq)')

def atribuir_faixa(nome_faixa, tabela_fonte, filtro_p, col_regra):
    if col_regra is None:
        filtro_regra = 'TRUE'
        regra_sql = "'score'"
        order_extra = '0'
    else:
        filtro_regra = f'{col_regra} IS NOT NULL'
        regra_sql = col_regra
        order_extra = 'desempate'

    con.execute(f'''
    CREATE OR REPLACE TABLE faixa_base AS
    SELECT
        id_seq_censo,
        id_seq_cpf,
        match_probability,
        {order_extra} AS desempate,
        {regra_sql} AS regra
    FROM {tabela_fonte}
    WHERE {filtro_p}
      AND {filtro_regra}
      AND id_seq_censo NOT IN (SELECT id_seq_censo FROM atribuicao)
      AND id_seq_cpf NOT IN (
          SELECT id_seq_cpf
          FROM atribuicao
          GROUP BY 1
          HAVING COUNT(*) >= {TETO}
      )
    ''')

    con.execute('''
    CREATE OR REPLACE TABLE faixa_melhor AS
    SELECT id_seq_censo, id_seq_cpf, match_probability, regra
    FROM faixa_base
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY id_seq_censo
        ORDER BY match_probability DESC, desempate DESC, id_seq_cpf
    ) = 1
    ''')

    antes = con.execute('SELECT COUNT(*) FROM atribuicao').fetchone()[0]
    con.execute(f'''
    INSERT INTO atribuicao
    SELECT
        m.id_seq_censo,
        m.id_seq_cpf,
        m.match_probability,
        '{nome_faixa}' AS faixa,
        m.regra
    FROM faixa_melhor m
    JOIN (
        SELECT id_seq_cpf, COUNT(*) AS n_novo
        FROM faixa_melhor
        GROUP BY 1
    ) g ON g.id_seq_cpf = m.id_seq_cpf
    LEFT JOIN (
        SELECT id_seq_cpf, COUNT(*) AS n_ja
        FROM atribuicao
        GROUP BY 1
    ) j ON j.id_seq_cpf = m.id_seq_cpf
    WHERE coalesce(j.n_ja, 0) + g.n_novo <= {TETO}
    ''')
    depois = con.execute('SELECT COUNT(*) FROM atribuicao').fetchone()[0]
    print(f'{nome_faixa} ({filtro_p}): +{depois - antes:,} (total {depois:,})')


## CSV da aplicação (`app_censo` / `app_cpf`)

Lê os CSV sem header do 07 (colunas Splink). Checkpoint parquet em `OUTPUT_DIR`.


In [ ]:
cols_sql = ', '.join(f"'{c}': 'VARCHAR'" for c in APP_COLS)

_tem_app = (
    duck_tem_tabela('app_censo')
    and duck_tem_tabela('app_cpf')
)
_tem_app_pq = APP_CENSO_OUT.exists() and APP_CPF_OUT.exists()

if not REBUILD_APP and _tem_app:
    print('app_censo/app_cpf: já no DuckDB')
elif not REBUILD_APP and _tem_app_pq:
    load_table_from_parquet(con, 'app_censo', APP_CENSO_OUT)
    load_table_from_parquet(con, 'app_cpf', APP_CPF_OUT)
    print('app_*: lido do parquet')
else:
    censo_p = str(CENSO_APP_CSV).replace(chr(39), chr(39) * 2)
    cpf_p = str(CPF_APP_CSV).replace(chr(39), chr(39) * 2)
    con.execute(f'''
    CREATE OR REPLACE TABLE app_censo AS
    SELECT * FROM read_csv(
        '{censo_p}',
        header=false,
        columns={{{cols_sql}}}
    )
    ''')
    con.execute(f'''
    CREATE OR REPLACE TABLE app_cpf AS
    SELECT * FROM read_csv(
        '{cpf_p}',
        header=false,
        columns={{{cols_sql}}}
    )
    ''')
    export_parquet(con, 'app_censo', path=APP_CENSO_OUT)
    export_parquet(con, 'app_cpf', path=APP_CPF_OUT)
    print('app_*: exportado')

n_censo = con.execute('SELECT COUNT(*) FROM app_censo').fetchone()[0]
n_cpf = con.execute('SELECT COUNT(*) FROM app_cpf').fetchone()[0]
print(f'app_censo: {n_censo:,} | app_cpf: {n_cpf:,}')
display(con.execute('SELECT * FROM app_censo LIMIT 3').df())
display(con.execute('SELECT * FROM app_cpf LIMIT 3').df())


## Predictions em `id_seq`

`COPY` do txt → `07b_pred_seq.parquet`. Confere se os `id_seq` existem no CSV app.


In [ ]:
_tipo = con.execute(
    "SELECT table_type FROM information_schema.tables "
    "WHERE table_schema = 'main' AND table_name = 'splink_predictions'"
).fetchone()

# Só dropa se for rebuild — senão reaproveita a tabela já no DuckDB.
if REBUILD_PRED and _tipo:
    _kind = 'VIEW' if _tipo[0].upper() == 'VIEW' else 'TABLE'
    con.execute(f'DROP {_kind} IF EXISTS splink_predictions')
    print('drop splink_predictions (REBUILD_PRED)')

for _t in ('pred_raw', 'pred_mapped'):
    if duck_tem_tabela(_t):
        con.execute(f'DROP TABLE IF EXISTS {_t}')
        print('drop:', _t)

# id1 = sempre censo, id2 = sempre cpf
id1_seq = "CASE WHEN id1 LIKE 'censo_%' THEN substr(id1, 7) ELSE id1 END"
id2_seq = "CASE WHEN id2 LIKE 'cpf_%' THEN substr(id2, 5) ELSE id2 END"

if not REBUILD_PRED and duck_tem_tabela('splink_predictions'):
    print('splink_predictions: já no DuckDB |',
          f"{con.execute('SELECT COUNT(*) FROM splink_predictions').fetchone()[0]:,}")
elif not REBUILD_PRED and PRED_SEQ_OUT.exists():
    load_table_from_parquet(con, 'splink_predictions', PRED_SEQ_OUT)
    print('splink_predictions: lido do parquet |',
          f"{con.execute('SELECT COUNT(*) FROM splink_predictions').fetchone()[0]:,}")
else:
    pred_path = str(PRED_TXT).replace("'", "''")
    out_path = str(PRED_SEQ_OUT).replace("'", "''")
    print('COPY streaming → 07b_pred_seq.parquet (id1=censo, id2=cpf) …')
    con.execute(f'''
    COPY (
        SELECT
            ({id1_seq}) AS id_seq_censo,
            ({id2_seq}) AS id_seq_cpf,
            TRY_CAST(match_probability AS DOUBLE) AS match_probability
        FROM read_csv(
            '{pred_path}',
            delim=' ',
            header=false,
            auto_detect=false,
            columns={{
                'status': 'VARCHAR',
                'x': 'VARCHAR',
                'id1': 'VARCHAR',
                'id2': 'VARCHAR',
                'total_score': 'VARCHAR',
                'match_probability': 'VARCHAR'
            }}
        )
        WHERE TRY_CAST(match_probability AS DOUBLE) >= {PISO_BAIXO}
    ) TO '{out_path}' (FORMAT PARQUET, COMPRESSION ZSTD)
    ''')
    print('exportado:', PRED_SEQ_OUT)
    load_table_from_parquet(con, 'splink_predictions', PRED_SEQ_OUT)
    print('splink_predictions: carregado |',
          f"{con.execute('SELECT COUNT(*) FROM splink_predictions').fetchone()[0]:,}")

n_pred = con.execute('SELECT COUNT(*) FROM splink_predictions').fetchone()[0]
print(f'pares: {n_pred:,}')

# Sanity: id_seq do pred existe no app
display(con.execute('''
SELECT
    COUNT(*) AS n_pred,
    COUNT(*) FILTER (
        WHERE ca.id_seq IS NOT NULL AND pb.id_seq IS NOT NULL
    ) AS n_com_app,
    COUNT(*) FILTER (WHERE ca.id_seq IS NULL) AS n_censo_fora_app,
    COUNT(*) FILTER (WHERE pb.id_seq IS NULL) AS n_cpf_fora_app
FROM splink_predictions p
LEFT JOIN app_censo ca ON ca.id_seq = p.id_seq_censo
LEFT JOIN app_cpf pb ON pb.id_seq = p.id_seq_cpf
''').df())

display(con.execute('''
SELECT
    CASE
        WHEN match_probability >= 0.97 THEN 'p_ge_097'
        WHEN match_probability >= 0.90 THEN 'p_090_097'
        WHEN match_probability >= 0.50 THEN 'p_050_090'
        WHEN match_probability >= 0.25 THEN 'p_025_050'
        ELSE 'p_lt_025'
    END AS faixa_p,
    COUNT(*) AS n
FROM splink_predictions
GROUP BY 1
ORDER BY 1 DESC
''').df())


## Atribuição — modelo

`p >= PISO_MODELO` (default 0.9894). Sem regras. Amostra de nomes do CSV para sanity.


In [ ]:
if REBUILD_ATRIB:
    con.execute('''
    CREATE OR REPLACE TABLE atribuicao (
        id_seq_censo VARCHAR,
        id_seq_cpf VARCHAR,
        match_probability DOUBLE,
        faixa VARCHAR,
        regra VARCHAR
    )
    ''')
    print('atribuicao: tabela vazia (REBUILD_ATRIB)')
else:
    garantir_atribuicao()

if not RUN_MODELO:
    print('RUN_MODELO=False — pulado')
else:
    con.execute(f'''
    CREATE OR REPLACE TABLE pred_modelo AS
    SELECT
        id_seq_censo,
        id_seq_cpf,
        match_probability
    FROM splink_predictions
    WHERE match_probability >= {PISO_MODELO}
      AND id_seq_censo IS NOT NULL
      AND id_seq_cpf IS NOT NULL
    ''')
    print('pred_modelo:', f"{con.execute('SELECT COUNT(*) FROM pred_modelo').fetchone()[0]:,}")

    atribuir_faixa(
        'modelo',
        'pred_modelo',
        f'match_probability >= {PISO_MODELO}',
        None,
    )
    export_parquet(con, 'atribuicao', path=ATRIB_SEQ_OUT)
    print('atribuicao id_seq (após modelo):', ATRIB_SEQ_OUT)

    # Sanity p alto: nomes do CSV devem parecer a mesma pessoa
    display(con.execute('''
    SELECT
        a.id_seq_censo,
        a.id_seq_cpf,
        a.match_probability,
        ca.nome_completo_phon AS nome_censo,
        pb.nome_completo_phon AS nome_cpf,
        ca.data_nascimento AS dn_censo,
        pb.data_nascimento AS dn_cpf
    FROM atribuicao a
    JOIN app_censo ca ON ca.id_seq = a.id_seq_censo
    JOIN app_cpf pb ON pb.id_seq = a.id_seq_cpf
    WHERE a.faixa = 'modelo'
    ORDER BY a.match_probability DESC
    LIMIT 15
    ''').df())


## Resgate — pares + atribuição

Join `app_censo` / `app_cpf` por `id_seq` só em `PISO_RESGATE <= p < PISO_MODELO`.


In [ ]:
tok_ca = "string_split(ca.nome_completo_phon, ' ')"
tok_pb = "string_split(pb.nome_completo_phon, ' ')"
n_ca = f"len({tok_ca})"
n_pb = f"len({tok_pb})"
n_min = f"least({n_ca}, {n_pb})"
n_inter = f"len(list_intersect({tok_ca}, {tok_pb}))"
cobertura = f"({n_inter} * 1.0 / nullif({n_min}, 0))"
contido = f"{n_min} >= 2 AND {n_ca} <> {n_pb} AND {n_inter} = {n_min}"
primeiro_ok = (
    "ca.primeiro_nome_phon IS NOT NULL AND pb.primeiro_nome_phon IS NOT NULL "
    "AND ("
    "ca.primeiro_nome_phon = pb.primeiro_nome_phon "
    "OR levenshtein(ca.primeiro_nome_phon, pb.primeiro_nome_phon) <= 1"
    ")"
)
data_igual = (
    "ca.data_nascimento IS NOT NULL AND pb.data_nascimento IS NOT NULL "
    "AND ca.data_nascimento = pb.data_nascimento"
)
data_ambas = (
    "ca.data_nascimento IS NOT NULL AND pb.data_nascimento IS NOT NULL "
    "AND ca.data_nascimento <> pb.data_nascimento"
)
# CSV app: data em DD/MM/YYYY (ex.: 18/01/1983)
dn_ca = "CAST(try_strptime(ca.data_nascimento, '%d/%m/%Y') AS DATE)"
dn_pb = "CAST(try_strptime(pb.data_nascimento, '%d/%m/%Y') AS DATE)"
delta_dias = f"abs(date_diff('day', {dn_ca}, {dn_pb}))"
mesmo_dia_mes = (
    f"day({dn_ca}) = day({dn_pb}) AND month({dn_ca}) = month({dn_pb})"
)
delta_ano = f"abs(year({dn_ca}) - year({dn_pb}))"
data_plausivel = (
    f"({delta_dias} <= 1) OR "
    f"(({mesmo_dia_mes}) AND ({delta_ano}) = 1) OR "
    f"(({mesmo_dia_mes}) AND ({delta_ano}) = 10)"
)
nome_muito_forte = (
    f"(ca.nome_completo_phon = pb.nome_completo_phon) OR ({contido})"
)
mae_presente = (
    "ca.nome_mae_phon IS NOT NULL AND pb.nome_mae_phon IS NOT NULL"
)
jw_mae = "jaro_winkler_similarity(ca.nome_mae_phon, pb.nome_mae_phon)"
r1_medio = (
    f"({data_igual}) AND ({primeiro_ok}) AND "
    f"(({contido}) OR ({cobertura} >= 0.75))"
)
r1_baixo = (
    f"({data_igual}) AND ({primeiro_ok}) AND "
    f"(({contido}) OR ({cobertura} >= 0.80))"
)
r2 = f"({nome_muito_forte}) AND ({data_ambas}) AND ({data_plausivel})"
r3_medio = (
    f"({primeiro_ok}) AND ({cobertura} >= 0.55) AND ({mae_presente}) "
    f"AND ({jw_mae} >= 0.75)"
)
desempate = (
    "("
    "CAST(ca.nome_mae_phon IS NOT NULL AND pb.nome_mae_phon IS NOT NULL AS INTEGER) "
    "+ CAST("
    "ca.logradouro_norm IS NOT NULL AND pb.logradouro_norm IS NOT NULL "
    "AND ca.logradouro_norm = pb.logradouro_norm AS INTEGER) "
    "+ CAST("
    "ca.cep IS NOT NULL AND pb.cep IS NOT NULL AND ca.cep = pb.cep AS INTEGER)"
    ")"
)



if not RUN_RESGATE:
    print('RUN_RESGATE=False — pulado')
else:
    garantir_atribuicao()

    if (
        not REBUILD_PARES_RESGATE
        and duck_tem_tabela('pares_resgate')
    ):
        print('pares_resgate: já no DuckDB |',
              f"{con.execute('SELECT COUNT(*) FROM pares_resgate').fetchone()[0]:,}")
    elif not REBUILD_PARES_RESGATE and PARES_RESGATE_OUT.exists():
        load_table_from_parquet(con, 'pares_resgate', PARES_RESGATE_OUT)
        print('pares_resgate: lido do parquet |',
              f"{con.execute('SELECT COUNT(*) FROM pares_resgate').fetchone()[0]:,}")
    else:
        con.execute(f'''
        CREATE OR REPLACE TABLE pares_resgate AS
        SELECT
            p.id_seq_censo,
            p.id_seq_cpf,
            p.match_probability,
            {desempate} AS desempate,
            CASE
                WHEN ({r1_medio}) THEN 'R1'
                WHEN ({r2}) THEN 'R2'
                WHEN ({r3_medio}) THEN 'R3'
            END AS regra_medio
        FROM splink_predictions p
        JOIN app_censo ca ON ca.id_seq = p.id_seq_censo
        JOIN app_cpf pb ON pb.id_seq = p.id_seq_cpf
        WHERE p.match_probability >= {PISO_RESGATE}
          AND p.match_probability < {PISO_MODELO}
        ''')
        n = con.execute('SELECT COUNT(*) FROM pares_resgate').fetchone()[0]
        print('pares_resgate:', f'{n:,}')
        display(con.execute('''
        SELECT
            COUNT(*) FILTER (WHERE regra_medio IS NOT NULL) AS n_com_regra,
            COUNT(*) FILTER (WHERE regra_medio = 'R1') AS n_r1,
            COUNT(*) FILTER (WHERE regra_medio = 'R2') AS n_r2,
            COUNT(*) FILTER (WHERE regra_medio = 'R3') AS n_r3
        FROM pares_resgate
        ''').df())
        export_parquet(con, 'pares_resgate', path=PARES_RESGATE_OUT)
        print('exportado:', PARES_RESGATE_OUT)

    atribuir_faixa(
        'resgate',
        'pares_resgate',
        (
            f'match_probability >= {PISO_RESGATE} '
            f'AND match_probability < {PISO_MODELO}'
        ),
        'regra_medio',
    )
    export_parquet(con, 'atribuicao', path=ATRIB_SEQ_OUT)
    print('atribuicao id_seq (após resgate):', ATRIB_SEQ_OUT)


## Baixo — pares + atribuição

Opcional (`RUN_BAIXO`).


In [ ]:
tok_ca = "string_split(ca.nome_completo_phon, ' ')"
tok_pb = "string_split(pb.nome_completo_phon, ' ')"
n_ca = f"len({tok_ca})"
n_pb = f"len({tok_pb})"
n_min = f"least({n_ca}, {n_pb})"
n_inter = f"len(list_intersect({tok_ca}, {tok_pb}))"
cobertura = f"({n_inter} * 1.0 / nullif({n_min}, 0))"
contido = f"{n_min} >= 2 AND {n_ca} <> {n_pb} AND {n_inter} = {n_min}"
primeiro_ok = (
    "ca.primeiro_nome_phon IS NOT NULL AND pb.primeiro_nome_phon IS NOT NULL "
    "AND ("
    "ca.primeiro_nome_phon = pb.primeiro_nome_phon "
    "OR levenshtein(ca.primeiro_nome_phon, pb.primeiro_nome_phon) <= 1"
    ")"
)
data_igual = (
    "ca.data_nascimento IS NOT NULL AND pb.data_nascimento IS NOT NULL "
    "AND ca.data_nascimento = pb.data_nascimento"
)
data_ambas = (
    "ca.data_nascimento IS NOT NULL AND pb.data_nascimento IS NOT NULL "
    "AND ca.data_nascimento <> pb.data_nascimento"
)
# CSV app: data em DD/MM/YYYY (ex.: 18/01/1983)
dn_ca = "CAST(try_strptime(ca.data_nascimento, '%d/%m/%Y') AS DATE)"
dn_pb = "CAST(try_strptime(pb.data_nascimento, '%d/%m/%Y') AS DATE)"
delta_dias = f"abs(date_diff('day', {dn_ca}, {dn_pb}))"
mesmo_dia_mes = (
    f"day({dn_ca}) = day({dn_pb}) AND month({dn_ca}) = month({dn_pb})"
)
delta_ano = f"abs(year({dn_ca}) - year({dn_pb}))"
data_plausivel = (
    f"({delta_dias} <= 1) OR "
    f"(({mesmo_dia_mes}) AND ({delta_ano}) = 1) OR "
    f"(({mesmo_dia_mes}) AND ({delta_ano}) = 10)"
)
nome_muito_forte = (
    f"(ca.nome_completo_phon = pb.nome_completo_phon) OR ({contido})"
)
mae_presente = (
    "ca.nome_mae_phon IS NOT NULL AND pb.nome_mae_phon IS NOT NULL"
)
jw_mae = "jaro_winkler_similarity(ca.nome_mae_phon, pb.nome_mae_phon)"
r1_medio = (
    f"({data_igual}) AND ({primeiro_ok}) AND "
    f"(({contido}) OR ({cobertura} >= 0.75))"
)
r1_baixo = (
    f"({data_igual}) AND ({primeiro_ok}) AND "
    f"(({contido}) OR ({cobertura} >= 0.80))"
)
r2 = f"({nome_muito_forte}) AND ({data_ambas}) AND ({data_plausivel})"
r3_medio = (
    f"({primeiro_ok}) AND ({cobertura} >= 0.55) AND ({mae_presente}) "
    f"AND ({jw_mae} >= 0.75)"
)
desempate = (
    "("
    "CAST(ca.nome_mae_phon IS NOT NULL AND pb.nome_mae_phon IS NOT NULL AS INTEGER) "
    "+ CAST("
    "ca.logradouro_norm IS NOT NULL AND pb.logradouro_norm IS NOT NULL "
    "AND ca.logradouro_norm = pb.logradouro_norm AS INTEGER) "
    "+ CAST("
    "ca.cep IS NOT NULL AND pb.cep IS NOT NULL AND ca.cep = pb.cep AS INTEGER)"
    ")"
)



if not RUN_BAIXO:
    print('RUN_BAIXO=False — pulado')
else:
    garantir_atribuicao()

    if not REBUILD_PARES_BAIXO and duck_tem_tabela('pares_baixo'):
        print('pares_baixo: já no DuckDB |',
              f"{con.execute('SELECT COUNT(*) FROM pares_baixo').fetchone()[0]:,}")
    elif not REBUILD_PARES_BAIXO and PARES_BAIXO_OUT.exists():
        load_table_from_parquet(con, 'pares_baixo', PARES_BAIXO_OUT)
        print('pares_baixo: lido do parquet |',
              f"{con.execute('SELECT COUNT(*) FROM pares_baixo').fetchone()[0]:,}")
    else:
        con.execute(f'''
        CREATE OR REPLACE TABLE pares_baixo AS
        SELECT
            p.id_seq_censo,
            p.id_seq_cpf,
            p.match_probability,
            {desempate} AS desempate,
            CASE
                WHEN ({r1_baixo}) THEN 'R1'
                WHEN ({r2}) THEN 'R2'
            END AS regra_baixo
        FROM splink_predictions p
        JOIN app_censo ca ON ca.id_seq = p.id_seq_censo
        JOIN app_cpf pb ON pb.id_seq = p.id_seq_cpf
        WHERE p.match_probability >= {PISO_BAIXO}
          AND p.match_probability < {PISO_RESGATE}
        ''')
        n = con.execute('SELECT COUNT(*) FROM pares_baixo').fetchone()[0]
        print('pares_baixo:', f'{n:,}')
        display(con.execute('''
        SELECT
            COUNT(*) FILTER (WHERE regra_baixo IS NOT NULL) AS n_com_regra,
            COUNT(*) FILTER (WHERE regra_baixo = 'R1') AS n_r1,
            COUNT(*) FILTER (WHERE regra_baixo = 'R2') AS n_r2
        FROM pares_baixo
        ''').df())
        export_parquet(con, 'pares_baixo', path=PARES_BAIXO_OUT)
        print('exportado:', PARES_BAIXO_OUT)

    atribuir_faixa(
        'baixo',
        'pares_baixo',
        (
            f'match_probability >= {PISO_BAIXO} '
            f'AND match_probability < {PISO_RESGATE}'
        ),
        'regra_baixo',
    )
    export_parquet(con, 'atribuicao', path=ATRIB_SEQ_OUT)
    print('atribuicao id_seq (após baixo):', ATRIB_SEQ_OUT)


## Cobertura (só aplicação CSV)


In [ ]:
n_app = con.execute('SELECT COUNT(*) FROM app_censo').fetchone()[0]
n_ok = con.execute('SELECT COUNT(*) FROM atribuicao').fetchone()[0]
print(f'Censo aplicação (CSV): {n_app:,}')
print(f'Atribuídos: {n_ok:,} ({100.0 * n_ok / n_app:.1f}%)')

display(con.execute('''
SELECT
    faixa,
    regra,
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM atribuicao
GROUP BY 1, 2
ORDER BY
    CASE faixa
        WHEN 'modelo' THEN 1
        WHEN 'resgate' THEN 2
        WHEN 'baixo' THEN 3
    END,
    regra
''').df())

display(con.execute('''
SELECT
    n_censos,
    COUNT(*) AS n_cpf,
    n_censos * COUNT(*) AS n_pares
FROM (
    SELECT id_seq_cpf, COUNT(*) AS n_censos
    FROM atribuicao
    GROUP BY 1
)
GROUP BY 1
ORDER BY 1
''').df())


## Cobertura por UF

`total_existente ≈ app_censo + lista ouro` (~91M + ~103M ≈ 195M). Disjuntos: ouro fora do CSV app.
- determinístico = lista ouro (UF via Luis só nos ids da lista)
- probabilístico = `atribuicao` (UF via `app_censo`)
- `total_pareado` = det + prob

In [ ]:
ouro_path = str(LISTA_OURO_AVALIACAO_ARQUIVO_TOTAL).replace("'", "''")
censo_luis = str(CENSO_V3_LUIS).replace("'", "''")

ouro_cols = {
    r[0] for r in con.execute(
        f"DESCRIBE SELECT * FROM read_parquet('{ouro_path}') LIMIT 0"
    ).fetchall()
}
id_ouro, _cpf_ouro = lista_ouro_colunas(ouro_cols)

con.execute(f'''
CREATE OR REPLACE TABLE lista_ouro AS
SELECT DISTINCT CAST({id_ouro} AS VARCHAR) AS id_censo
FROM read_parquet('{ouro_path}')
WHERE {id_ouro} IS NOT NULL
''')
n_det = con.execute('SELECT COUNT(*) FROM lista_ouro').fetchone()[0]
n_app = con.execute('SELECT COUNT(*) FROM app_censo').fetchone()[0]
print(f'lista ouro (det): {n_det:,} | app_censo: {n_app:,} | soma: {n_det + n_app:,}')

# det: Luis só nos ids da lista (UF). prob: app + atrib.
# total_existente = det + app (disjuntos). total_pareado = det + prob.
con.execute(f'''
CREATE OR REPLACE TABLE cobertura_uf AS
WITH det_uf AS (
    SELECT
        substr(CAST(l.cod_mun AS VARCHAR), 1, 2) AS uf,
        COUNT(*) AS deterministico
    FROM read_parquet('{censo_luis}') l
    WHERE CAST(l.id_morador AS VARCHAR) IN (SELECT id_censo FROM lista_ouro)
    GROUP BY 1
),
prob_uf AS (
    SELECT
        substr(c.cod_municipio, 1, 2) AS uf,
        COUNT(*) FILTER (WHERE a.id_seq_censo IS NOT NULL) AS probabilistico,
        COUNT(*) AS n_app_uf
    FROM app_censo c
    LEFT JOIN atribuicao a ON a.id_seq_censo = c.id_seq
    GROUP BY 1
)
SELECT
    coalesce(d.uf, p.uf) AS uf,
    coalesce(d.deterministico, 0) AS deterministico,
    coalesce(p.probabilistico, 0) AS probabilistico,
    coalesce(d.deterministico, 0) + coalesce(p.probabilistico, 0) AS total_pareado,
    coalesce(d.deterministico, 0) + coalesce(p.n_app_uf, 0) AS total_existente,
    ROUND(
        100.0 * (coalesce(d.deterministico, 0) + coalesce(p.probabilistico, 0))
        / nullif(coalesce(d.deterministico, 0) + coalesce(p.n_app_uf, 0), 0),
        2
    ) AS pct
FROM det_uf d
FULL OUTER JOIN prob_uf p ON p.uf = d.uf
ORDER BY 1
''')

display(con.execute('SELECT * FROM cobertura_uf').df())
display(con.execute('''
SELECT
    SUM(deterministico) AS deterministico,
    SUM(probabilistico) AS probabilistico,
    SUM(total_pareado) AS total_pareado,
    SUM(total_existente) AS total_existente,
    ROUND(
        100.0 * SUM(total_pareado) / nullif(SUM(total_existente), 0),
        2
    ) AS pct
FROM cobertura_uf
''').df())

export_parquet(con, 'cobertura_uf', path=COBERTURA_UF_OUT)
print('Exportado:', COBERTURA_UF_OUT)


## Qualidade dos pares atribuídos

Acordo via campos do CSV da aplicação (não Luis).


In [ ]:
display(con.execute('''
SELECT
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE ca.nome_mae_phon IS NOT NULL AND pb.nome_mae_phon IS NOT NULL
          AND trim(ca.nome_mae_phon) <> '' AND trim(pb.nome_mae_phon) <> ''
    ) / COUNT(*), 1) AS pct_mae_preenchida,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE ca.nome_mae_phon IS NOT NULL AND pb.nome_mae_phon IS NOT NULL
          AND trim(ca.nome_mae_phon) <> '' AND trim(pb.nome_mae_phon) <> ''
          AND ca.nome_mae_phon = pb.nome_mae_phon
    ) / COUNT(*), 1) AS pct_mae_igual,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE ca.logradouro_norm IS NOT NULL AND pb.logradouro_norm IS NOT NULL
          AND trim(ca.logradouro_norm) <> '' AND trim(pb.logradouro_norm) <> ''
          AND ca.logradouro_norm = pb.logradouro_norm
    ) / COUNT(*), 1) AS pct_logradouro_igual,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE ca.cep IS NOT NULL AND pb.cep IS NOT NULL
          AND trim(ca.cep) <> '' AND trim(pb.cep) <> ''
          AND ca.cep = pb.cep
    ) / COUNT(*), 1) AS pct_cep_igual,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE ca.data_nascimento IS NOT NULL AND pb.data_nascimento IS NOT NULL
          AND ca.data_nascimento = pb.data_nascimento
    ) / COUNT(*), 1) AS pct_data_igual,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE substr(ca.cod_municipio, 1, 2) = substr(pb.cod_municipio, 1, 2)
    ) / COUNT(*), 1) AS pct_uf_igual
FROM atribuicao a
JOIN app_censo ca ON ca.id_seq = a.id_seq_censo
JOIN app_cpf pb ON pb.id_seq = a.id_seq_cpf
''').df())

display(con.execute('''
SELECT
    a.faixa,
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE ca.nome_mae_phon IS NOT NULL AND pb.nome_mae_phon IS NOT NULL
          AND trim(ca.nome_mae_phon) <> '' AND trim(pb.nome_mae_phon) <> ''
          AND ca.nome_mae_phon = pb.nome_mae_phon
    ) / COUNT(*), 1) AS pct_mae_igual,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE ca.data_nascimento IS NOT NULL AND pb.data_nascimento IS NOT NULL
          AND ca.data_nascimento = pb.data_nascimento
    ) / COUNT(*), 1) AS pct_data_igual,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE ca.cep IS NOT NULL AND pb.cep IS NOT NULL
          AND trim(ca.cep) <> '' AND trim(pb.cep) <> ''
          AND ca.cep = pb.cep
    ) / COUNT(*), 1) AS pct_cep_igual,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE ca.logradouro_norm IS NOT NULL AND pb.logradouro_norm IS NOT NULL
          AND trim(ca.logradouro_norm) <> '' AND trim(pb.logradouro_norm) <> ''
          AND ca.logradouro_norm = pb.logradouro_norm
    ) / COUNT(*), 1) AS pct_logradouro_igual,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE substr(ca.cod_municipio, 1, 2) = substr(pb.cod_municipio, 1, 2)
    ) / COUNT(*), 1) AS pct_uf_igual
FROM atribuicao a
JOIN app_censo ca ON ca.id_seq = a.id_seq_censo
JOIN app_cpf pb ON pb.id_seq = a.id_seq_cpf
GROUP BY 1
ORDER BY
    CASE a.faixa
        WHEN 'modelo' THEN 1
        WHEN 'resgate' THEN 2
        WHEN 'baixo' THEN 3
    END
''').df())

display(con.execute('''
SELECT
    faixa,
    COUNT(*) AS n,
    ROUND(MIN(match_probability), 4) AS p_min,
    ROUND(quantile_cont(match_probability, 0.10), 4) AS p_p10,
    ROUND(quantile_cont(match_probability, 0.50), 4) AS p_p50,
    ROUND(quantile_cont(match_probability, 0.90), 4) AS p_p90,
    ROUND(MAX(match_probability), 4) AS p_max
FROM atribuicao
GROUP BY 1
ORDER BY
    CASE faixa
        WHEN 'modelo' THEN 1
        WHEN 'resgate' THEN 2
        WHEN 'baixo' THEN 3
    END
''').df())


## Quem ficou de fora

Residual = censo do app sem atribuição. Compara preenchimento de mãe / data / CEP / logradouro
(atribuído vs residual) e, no residual, o mesmo por faixa do melhor par no pred.


In [ ]:
# Residual = censo do app sem linha em atribuicao.
# faixa_p = melhor par em splink_predictions (mesmo sem ter entrado na escada).
con.execute('''
CREATE OR REPLACE TABLE melhor_qualquer AS
SELECT
    id_seq_censo,
    id_seq_cpf,
    match_probability
FROM splink_predictions
WHERE id_seq_censo IS NOT NULL AND id_seq_cpf IS NOT NULL
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY id_seq_censo
    ORDER BY match_probability DESC, id_seq_cpf
) = 1
''')

con.execute('''
CREATE OR REPLACE TABLE censo_sem_cpf AS
SELECT c.id_seq AS id_seq_censo
FROM app_censo c
WHERE NOT EXISTS (
    SELECT 1 FROM atribuicao a WHERE a.id_seq_censo = c.id_seq
)
''')

con.execute(f'''
CREATE OR REPLACE TABLE sem_cpf_melhor AS
SELECT
    s.id_seq_censo,
    m.id_seq_cpf,
    m.match_probability,
    CASE
        WHEN m.match_probability IS NULL THEN 'sem_par'
        WHEN m.match_probability >= {PISO_MODELO} THEN 'p_ge_piso'
        WHEN m.match_probability >= 0.90 THEN 'p_090_piso'
        WHEN m.match_probability >= {PISO_RESGATE} THEN 'p_050_090'
        WHEN m.match_probability >= {PISO_BAIXO} THEN 'p_025_050'
        ELSE 'p_lt_025'
    END AS faixa_p
FROM censo_sem_cpf s
LEFT JOIN melhor_qualquer m ON m.id_seq_censo = s.id_seq_censo
''')

n_sem = con.execute('SELECT COUNT(*) FROM censo_sem_cpf').fetchone()[0]
assert n_ok + n_sem == n_app
print(f'residual (sem atribuição): {n_sem:,} de {n_app:,} ({100.0 * n_sem / n_app:.1f}%)')

# Perfil de preenchimento no Censo: atribuído vs residual
display(con.execute('''
SELECT
    CASE WHEN a.id_seq_censo IS NOT NULL THEN 'atribuido' ELSE 'residual' END AS grupo,
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE c.nome_mae_phon IS NULL OR trim(c.nome_mae_phon) = ''
    ) / COUNT(*), 1) AS pct_sem_mae,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE c.data_nascimento IS NULL OR trim(CAST(c.data_nascimento AS VARCHAR)) = ''
    ) / COUNT(*), 1) AS pct_sem_data,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE c.cep IS NULL OR trim(c.cep) = ''
    ) / COUNT(*), 1) AS pct_sem_cep,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE c.logradouro_norm IS NULL OR trim(c.logradouro_norm) = ''
    ) / COUNT(*), 1) AS pct_sem_logradouro
FROM app_censo c
LEFT JOIN atribuicao a ON a.id_seq_censo = c.id_seq
GROUP BY 1
ORDER BY 1
''').df())

# No residual: mesmo perfil por faixa do melhor par
display(con.execute('''
SELECT
    s.faixa_p,
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_residual,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE c.nome_mae_phon IS NULL OR trim(c.nome_mae_phon) = ''
    ) / COUNT(*), 1) AS pct_sem_mae,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE c.data_nascimento IS NULL OR trim(CAST(c.data_nascimento AS VARCHAR)) = ''
    ) / COUNT(*), 1) AS pct_sem_data,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE c.cep IS NULL OR trim(c.cep) = ''
    ) / COUNT(*), 1) AS pct_sem_cep,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE c.logradouro_norm IS NULL OR trim(c.logradouro_norm) = ''
    ) / COUNT(*), 1) AS pct_sem_logradouro
FROM sem_cpf_melhor s
JOIN app_censo c ON c.id_seq = s.id_seq_censo
GROUP BY 1
ORDER BY
    CASE s.faixa_p
        WHEN 'p_ge_piso' THEN 1
        WHEN 'p_090_piso' THEN 2
        WHEN 'p_050_090' THEN 3
        WHEN 'p_025_050' THEN 4
        WHEN 'p_lt_025' THEN 5
        WHEN 'sem_par' THEN 6
    END
''').df())


In [ ]:
con.close()
print('ok')
